# 3.  Codificación de variables

Conjunto de datos: titanic (incluido en seaborn)

Este conjunto de datos es excelente para trabajar con variables categóricas.

In [25]:
# PASO 1: Importación de librerías y parche SSL
import ssl

# 'ssl._create_unverified_context' desactiva la validación estricta de certificados HTTPS en Python 3.12 (macOS).
# Esto evita que urllib o seaborn arrojen el error 'SSLCertVerificationError' al intentar descargar conjuntos de datos desde la red.
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd                                  # Librería para crear y manipular estructuras de datos bidimensionales (DataFrames).
import seaborn as sns                                # Librería de visualización que incluye datasets de prueba en formato Pandas.
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # Clases de Scikit-Learn especializadas en transformar variables categóricas (texto) a números.
import matplotlib.pyplot as plt

In [26]:
# PASO 2: Carga del dataset Titanic
# 'sns.load_dataset('titanic')' descarga el archivo desde el repositorio oficial de Seaborn
# y lo convierte directamente en un objeto de tipo DataFrame de Pandas guardado en la variable 'data'.
# Incluye un bloque 'try-except' para leer una copia raw en la web en caso de un bloqueo de red local.
try:
    data = sns.load_dataset('titanic')
except Exception:
    data = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

In [27]:
# PASO 3: Inspección visual inicial
# 'data.head()' extrae las primeras 5 filas (índices 0 al 4) de la tabla 'data' con todas sus columnas originales.
# Sirve para inspeccionar qué columnas contienen texto (categóricas), números enteros/flotantes y valores faltantes (NaN).
print("Conjunto de datos Titanic:")
print(data.head())

Conjunto de datos Titanic:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [28]:
# PASO 4: Identificación de variables categóricas
# Guardamos en una lista de Python los nombres exactos de las columnas de tipo texto/objeto que queremos procesar.
# 'sex' (género: male/female), 'embarked' (puerto: S/C/Q) y 'class' (clase social: First/Second/Third).
categorical_cols = ['sex', 'embarked', 'class']

In [29]:
# PASO 5: Codificación ordinal numérica con LabelEncoder
# 1. 'LabelEncoder()' crea un objeto transformador que asigna un número entero único (0, 1, 2...) a cada categoría de texto según su orden alfabético.
label_encoder = LabelEncoder()

# 2. 'label_encoder.fit_transform(data['sex'])':
#    - 'fit': Escanea la columna 'sex' e identifica las categorías únicas presentes (en este caso: 'female' y 'male').
#    - Mapea alfabéticamente: 'female' se convierte en el número entero 0, y 'male' se convierte en el número entero 1.
#    - 'transform': Reemplaza todo texto de la columna por su entero asignado y retorna un arreglo numérico unidimensional (NumPy array).
# 3. 'data['sex_label'] = ...': Crea una nueva columna en la tabla llamada 'sex_label' donde deposita esos valores 0 y 1.
data['sex_label'] = label_encoder.fit_transform(data['sex'])

In [30]:
# PASO 6: Codificación binaria independiente (One-Hot Encoding)
# 'pd.get_dummies()' evalúa la columna 'embarked' (que contiene 3 categorías de texto: 'S', 'C' y 'Q').
# En lugar de usar números 0, 1 y 2 en una sola columna, genera 3 columnas independientes conteniendo únicamente valores 1 o 0 (verdadero/falso):
#  - 'embarked_C': 1 si abordó en Cherbourg, 0 si no.
#  - 'embarked_Q': 1 si abordó en Queenstown, 0 si no.
#  - 'embarked_S': 1 si abordó en Southampton, 0 si no.
# El parámetro 'prefix='embarked'' antepone esa palabra al nombre de las 3 nuevas columnas resultantes.
onehot_encoder = pd.get_dummies(data['embarked'], prefix='embarked')

In [31]:
# PASO 7: Unificación de estructuras de datos
# 'pd.concat([data, onehot_encoder], axis=1)' toma la tabla principal ('data') y le pega al lado derecho (horizontalmente, por eso axis=1)
# las 3 nuevas columnas binarias generadas en 'onehot_encoder' ('embarked_C', 'embarked_Q', 'embarked_S').
# El resultado actualiza la variable 'data' con la tabla expandida.
data = pd.concat([data, onehot_encoder], axis=1)

# Ejercicios
Contesta las siguientes preguntas. Para cada pregunta, deberás escribir código que demostrará cómo llegaste al resultado. Crea gráficas en donde veas correcto.

### 1. ¿Qué diferencias encuentras entre LabelEncoder y OneHotEncoder?

LabelEncoder le asigna un número a cada categoría en una sola columna (0, 1, 2...). OneHotEncoder crea una columna nueva de 0 y 1 para cada categoría.

### 2. Crea una gráfica de barras comparando las frecuencias de 'sex' antes y después de la codificación con LabelEncoder.

In [ ]:
# Creamos un lienzo con 2 subgráficas lado a lado (1 fila, 2 columnas)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Subgráfica 1: Frecuencias de la columna original con texto ('male', 'female')
data['sex'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Frecuencia Original (sex)')
axes[0].set_xlabel('Categoría (Texto)')
axes[0].set_ylabel('Cantidad de Pasajeros')
axes[0].tick_params(axis='x', rotation=0)

# Subgráfica 2: Frecuencias de la columna codificada con enteros (1 y 0)
data['sex_label'].value_counts().plot(kind='bar', ax=axes[1], color='orange', edgecolor='black')
axes[1].set_title('Frecuencia Codificada (sex_label)')
axes[1].set_xlabel('Etiqueta Numérica (0 = female, 1 = male)')
axes[1].set_ylabel('Cantidad de Pasajeros')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3. Utiliza OneHotEncoder para codificar la columna 'class'. ¿Qué ventajas tiene este enfoque frente a LabelEncoder?

In [ ]:

class_dummies = pd.get_dummies(data['class'], prefix='class')
data = pd.concat([data, class_dummies], axis=1)

### 4. Si quisieras aplicar un modelo de aprendizaje automático, ¿qué tipo de codificación elegirías para las variables categóricas? Explica tu respuesta.

Usaría OneHotEncoder para variables sin orden, como la columna género o puerto de embarque, para no inventar jerarquías falsas, y LabelEncoder solo cuando las categorías tengan un orden lógico claro, como las columnas nivel bajo, medio y alto)